# Qalqan Audio Trigger — Colab (EN, с восстановлением)

Модель A для английского «help» (S15). Ноутбук умеет продолжать после перезапуска: код — из `qalqan_*.zip`, готовые данные — из `prepared.tar` на Диске (если есть), остальное скачивается с проверкой хешей.

GPU (T4) нужен только для обучения (раздел 7); для синтеза он не помогает.

## 1. Диск и код проекта
В `MyDrive/qalqan` должен лежать свежий `qalqan_*.zip`. Проверьте в выводе, что имя архива — самое новое.

In [ ]:
import glob
import os

from google.colab import drive

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/qalqan'
bundle = max(glob.glob(f'{DRIVE_DIR}/qalqan_*.zip'), key=os.path.getmtime)
print('архив:', bundle)
!unzip -q -o "$bundle" -d /content
os.chdir('/content/qalqan')
print('ядер CPU:', os.cpu_count())

## 2. Восстановить сохранённое (если есть)
`prepared.tar` — клипы FSD50K, голоса Piper, Vosk, тексты FLEURS. Если файла нет, ячейка ничего не делает.

In [ ]:
prepared = f'{DRIVE_DIR}/prepared.tar'
if os.path.isfile(prepared):
    !tar -xf "$prepared"
    print('восстановлено из', prepared)
else:
    print('prepared.tar нет — всё скачается заново')
have_fsd = os.path.isfile('data/processed/fsd50k_eval/manifest_sed_neg3000_seed0.jsonl')
PROFILE = 'kws_core' if have_fsd else 'kws'
print('профиль ассетов:', PROFILE)

## 3. Установка и недостающие ассеты
Профиль `kws_core` не качает FSD50K (6,3 ГБ), если готовые клипы уже восстановлены.

In [ ]:
!bash scripts/colab_setup.sh $PROFILE

In [ ]:
# Разметка FSD50K (335 КБ) нужна сборке набора; если её нет — докачать (только она, md5)
!uv run python -m qat.assets fetch-fsd-gt

## 4. Проверка: тесты проекта

In [ ]:
!uv run pytest -q 2>&1 | tail -3

## 5. Сохранить подготовленное на Диск (один раз)
Если `prepared.tar` на Диске уже есть, ячейка его не трогает.

In [ ]:
if not os.path.isfile(prepared):
    !tar -cf prepared.tar data/processed/fsd50k_eval models/piper models/vosk data/kscd data/census
    !cp prepared.tar "$DRIVE_DIR"/ && ls -lh "$DRIVE_DIR"
else:
    print('prepared.tar уже на Диске')

## 6. S11b: синтез (CPU, самая долгая часть) и сборка набора
В начале вывода: «запусков Piper: 72». Каждая строка «готово» — один запуск. Не закрывайте вкладку.

In [ ]:
!uv run python -m qat.data.kws_dataset synth

In [ ]:
!uv run python -m qat.data.kws_dataset build

Сразу сохранить набор на Диск — `/content` стирается при перезапуске.

In [ ]:
!tar -czf kws_dataset.tar.gz data/processed/kws
!cp kws_dataset.tar.gz "$DRIVE_DIR"/ && ls -lh "$DRIVE_DIR"

## 7. S12: обучение модели A (T4: *Среда выполнения → Сменить среду → T4*)
Смена среды стирает `/content`: если переключаетесь, сначала сохраните набор (ячейка выше), потом после перезапуска выполните разделы 1–3 и `tar -xzf` набора:

`!tar -xzf "$DRIVE_DIR"/kws_dataset.tar.gz`

In [ ]:
!uv run python -m qat.train

## 8. Сохранить модель на Диск
На Мак нужны `kws_dscnn_v1.pt` и отчёт — в `models/kws/`.

In [ ]:
!mkdir -p "$DRIVE_DIR"/model
!cp models/kws/kws_dscnn_v1.pt models/kws/kws_dscnn_v1_report.json "$DRIVE_DIR"/model/
!ls -lh "$DRIVE_DIR"/model